# 진짜 Stacking 메타러너 (중첩 CV 방식)

지금까지 한 "블렌딩"은 전부 가중평균(단순 선형결합)이었습니다. 이 노트북은
**진짜 Stacking**을 합니다: 4개 베이스 모델의 OOF 예측값 자체를 입력 피처로
삼아서, 2단계 메타모델(로지스틱회귀 / 얕은 LightGBM)을 새로 학습시킵니다.

**누수 방지**: 베이스 모델 OOF를 만들 때 쓴 것과 동일한 5-Fold 분할
(`random_state=1004`)을 메타러너 학습에도 그대로 재사용합니다. 이렇게 하면
메타러너가 각 fold를 예측할 때, 그 fold의 베이스 OOF 값들은 이미 "그 fold를
한 번도 본 적 없는 베이스 모델"이 만든 것이고, 메타러너 자신도 그 fold를
학습에 쓰지 않으니 이중으로 깨끗합니다.

numpy/sklearn/lightgbm만 쓰는 가벼운 노트북입니다.

**사전 준비**: `blend_cache/`에 다음 파일들이 있어야 합니다.
- `oof_lgbm_tuned.npy`, `oof_catboost.npy`, `oof_xgboost.npy`, `oof_tt.npy`, `oof_y.npy`

## 1. 데이터 로드 및 메타 피처 구성

In [1]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

CACHE_DIR = "blend_cache"
RANDOM_STATE = 1004
N_SPLITS = 5

oof_lgbm = np.load(f"{CACHE_DIR}/oof_lgbm_tuned.npy")
oof_cat = np.load(f"{CACHE_DIR}/oof_catboost.npy")
oof_xgb = np.load(f"{CACHE_DIR}/oof_xgboost.npy")
oof_tt = np.load(f"{CACHE_DIR}/oof_tt.npy")
y = np.load(f"{CACHE_DIR}/oof_y.npy")

meta_X = np.column_stack([oof_lgbm, oof_cat, oof_xgb, oof_tt])
print(f"메타 피처 shape: {meta_X.shape}  (4개 베이스 모델의 OOF 예측값)")

base_names = ["LightGBM", "CatBoost", "XGBoost", "TabTransformer"]
for i, name in enumerate(base_names):
    print(f"  {name} 단독 AUC: {roc_auc_score(y, meta_X[:, i]):.5f}")

메타 피처 shape: (256351, 4)  (4개 베이스 모델의 OOF 예측값)
  LightGBM 단독 AUC: 0.73998
  CatBoost 단독 AUC: 0.73971
  XGBoost 단독 AUC: 0.73972
  TabTransformer 단독 AUC: 0.73523


## 2. 베이스라인: 단순 가중평균 (비교 기준)

In [2]:
# 4개 전부 동일 가중치 평균 (참고용 베이스라인)
simple_avg = meta_X.mean(axis=1)
simple_avg_score = roc_auc_score(y, simple_avg)
print(f"4개 단순평균 AUC: {simple_avg_score:.5f}")

# 우리가 이미 확인한 LightGBM+CatBoost+XGBoost 3종 가중평균 최적값(참고)
best_individual = max(roc_auc_score(y, meta_X[:, i]) for i in range(meta_X.shape[1]))
print(f"4개 모델 중 단독 최고: {best_individual:.5f}")

4개 단순평균 AUC: 0.74028
4개 모델 중 단독 최고: 0.73998


## 3. Stacking (메타러너 1: 로지스틱 회귀)

In [3]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
meta_oof_lr = np.zeros(len(y))

for tr_idx, val_idx in skf.split(meta_X, y):
    meta_model = LogisticRegression(max_iter=1000)
    meta_model.fit(meta_X[tr_idx], y[tr_idx])
    meta_oof_lr[val_idx] = meta_model.predict_proba(meta_X[val_idx])[:, 1]

score_lr = roc_auc_score(y, meta_oof_lr)
print(f"Stacking (로지스틱회귀 메타러너) AUC: {score_lr:.5f}")
print(f"단독 최고 대비 개선: {score_lr - best_individual:+.5f}")

# 각 fold에서 학습된 마지막 메타모델의 계수도 참고로 출력 (어느 베이스모델을 더 신뢰하는지)
print("\n(참고) 마지막 fold 메타모델이 각 베이스모델에 준 가중치(계수):")
for name, coef in zip(base_names, meta_model.coef_[0]):
    print(f"  {name}: {coef:.4f}")

Stacking (로지스틱회귀 메타러너) AUC: 0.74019
단독 최고 대비 개선: +0.00021

(참고) 마지막 fold 메타모델이 각 베이스모델에 준 가중치(계수):
  LightGBM: 1.5652
  CatBoost: 1.0981
  XGBoost: 1.4942
  TabTransformer: 2.3024


## 4. Stacking (메타러너 2: 얕은 LightGBM, 과적합 방지용으로 강하게 정규화)

In [4]:
meta_oof_lgbm = np.zeros(len(y))

for tr_idx, val_idx in skf.split(meta_X, y):
    meta_model = LGBMClassifier(
        max_depth=2,
        num_leaves=4,
        n_estimators=50,
        learning_rate=0.05,
        random_state=RANDOM_STATE,
        verbose=-1,
    )
    meta_model.fit(meta_X[tr_idx], y[tr_idx])
    meta_oof_lgbm[val_idx] = meta_model.predict_proba(meta_X[val_idx])[:, 1]

score_lgbm_meta = roc_auc_score(y, meta_oof_lgbm)
print(f"Stacking (얕은 LightGBM 메타러너) AUC: {score_lgbm_meta:.5f}")
print(f"단독 최고 대비 개선: {score_lgbm_meta - best_individual:+.5f}")

/Users/sojung/Desktop/AI_Health_care/17. 난임 환자 심신 성공 여부 예측 AI 해커톤/git/infertility_classification/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/sojung/Desktop/AI_Health_care/17. 난임 환자 심신 성공 여부 예측 AI 해커톤/git/infertility_classification/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/sojung/Desktop/AI_Health_care/17. 난임 환자 심신 성공 여부 예측 AI 해커톤/git/infertility_classification/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Stacking (얕은 LightGBM 메타러너) AUC: 0.73981
단독 최고 대비 개선: -0.00017


/Users/sojung/Desktop/AI_Health_care/17. 난임 환자 심신 성공 여부 예측 AI 해커톤/git/infertility_classification/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/sojung/Desktop/AI_Health_care/17. 난임 환자 심신 성공 여부 예측 AI 해커톤/git/infertility_classification/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 5. 종합 비교 및 결론

In [5]:
print("=" * 60)
print(f"{'방법':40s} AUC")
print("-" * 60)
print(f"{'단독 최고':40s} {best_individual:.5f}")
print(f"{'단순 4개 평균':40s} {simple_avg_score:.5f}")
print(f"{'Stacking (로지스틱회귀)':40s} {score_lr:.5f}")
print(f"{'Stacking (얕은 LightGBM)':40s} {score_lgbm_meta:.5f}")
print("=" * 60)

best_method_score = max(simple_avg_score, score_lr, score_lgbm_meta)
improvement = best_method_score - best_individual

if improvement > 0.001:
    print(f"\n=> 개선 {improvement:+.5f}, 노이즈를 넘는 의미 있는 결과입니다! 제출 후보로 검토해보세요.")
else:
    print(f"\n=> 개선 {improvement:+.5f}, 여전히 노이즈 수준입니다.")
    print("   진짜 Stacking도 단순 가중평균과 큰 차이가 없습니다 — 베이스 모델들이")
    print("   이미 서로 너무 비슷한 정보를 담고 있어서, 메타러너가 더 짜낼 게 없는 것으로 보입니다.")

방법                                       AUC
------------------------------------------------------------
단독 최고                                    0.73998
단순 4개 평균                                 0.74028
Stacking (로지스틱회귀)                        0.74019
Stacking (얕은 LightGBM)                   0.73981

=> 개선 +0.00031, 여전히 노이즈 수준입니다.
   진짜 Stacking도 단순 가중평균과 큰 차이가 없습니다 — 베이스 모델들이
   이미 서로 너무 비슷한 정보를 담고 있어서, 메타러너가 더 짜낼 게 없는 것으로 보입니다.
